In [19]:
import json
import pandas as pd
import os

In [20]:
Test_matches_list = []
Test_deliveries_list = []

In [22]:
data_path = r"C:\Users\LENOVO\OneDrive\Desktop\Cricsheet - Analysis\data_raw\tests"

files = os.listdir(data_path)
print("Total files:", len(files))

Total files: 3086


In [23]:
Test_matches_list = []
Test_deliveries_list = []

In [39]:
import os
import json

Test_matches_data = []
Test_deliveries_data = []

folder_path = r"C:\Users\LENOVO\OneDrive\Desktop\Cricsheet - Analysis\data_raw\tests"

for file in os.listdir(folder_path):

    if not file.endswith(".json"):
        continue

    file_path = os.path.join(folder_path, file)

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # ---------------------------
    # ✅ MATCH LEVEL DATA (ONCE)
    # ---------------------------
    info = data.get("info", {})

    match_id = file.replace(".json", "")
    teams = info.get("teams", ["NA", "NA"])

    team1 = teams[0]
    team2 = teams[1]

    venue = info.get("venue", "Unknown")
    city = info.get("city", "Unknown")
    date = info.get("dates", ["Unknown"])[0]

    Test_matches_data.append({
        "match_id": match_id,
        "team1": team1,
        "team2": team2,
        "venue": venue,
        "city": city,
        "date": date,
        "match_type": "ODI"
    })

    # ---------------------------
    # ✅ DELIVERY LEVEL DATA
    # ---------------------------
    for innings in data["innings"]:

        # Case 1: Nested format ("1st innings")
        if isinstance(innings, dict) and len(innings.keys()) == 1:
            inning_name = list(innings.keys())[0]
            inning_data = innings[inning_name]

        # Case 2: Flat format (team + overs directly)
        else:
            inning_name = innings.get("team", "unknown")
            inning_data = innings

        # Skip if no ball data
        if "overs" not in inning_data:
            continue

        for over in inning_data["overs"]:
            over_number = over["over"]

            for ball_index, delivery in enumerate(over["deliveries"], start=1):

                batter = delivery.get("batter", "")
                bowler = delivery.get("bowler", "")

                runs_info = delivery.get("runs", {})
                runs = runs_info.get("batter", 0)
                extras = runs_info.get("extras", 0)
                total = runs_info.get("total", 0)

                wicket = 1 if "wickets" in delivery else 0

                Test_deliveries_data.append({
                    "match_id": match_id,
                    "innings": inning_name,
                    "over": over_number,
                    "ball": ball_index,   # ✅ generated value
                    "batter": batter,
                    "bowler": bowler,
                    "runs_batter": runs,
                    "extras": extras,
                    "total_runs": total,
                    "is_wicket": wicket
                })

In [40]:
print("Matches:", len(Test_matches_data))
print("Deliveries:", len(Test_deliveries_data))

Matches: 3085
Deliveries: 1632502


In [41]:
matches_df = pd.DataFrame(Test_matches_data)
deliveries_df = pd.DataFrame(Test_deliveries_data)

In [42]:
matches_df.to_csv(r"C:\Users\LENOVO\OneDrive\Desktop\Cricsheet - Analysis\data_processed\Test_matches.csv", index=False)
deliveries_df.to_csv(r"C:\Users\LENOVO\OneDrive\Desktop\Cricsheet - Analysis\data_processed\Test_deliveries.csv", index=False)